# Milestone 5

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
LABELS = ['A','B','C','D','E']
label2idx = {l:i for i,l in enumerate(LABELS)}
idx2label = {i:l for i,l in enumerate(LABELS)}

## Setup: Load Fine-Tuned Models

In [ ]:
# Load fine-tuned checkpoints
deberta_model = AutoModelForSequenceClassification.from_pretrained('microsoft/deberta-v3-small', num_labels=5)
deberta_tok = AutoTokenizer.from_pretrained('microsoft/deberta-v3-small')

roberta_model = AutoModelForSequenceClassification.from_pretrained('roberta-base', num_labels=5)
roberta_tok = AutoTokenizer.from_pretrained('roberta-base')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
deberta_model = deberta_model.to(device).eval()
roberta_model = roberta_model.to(device).eval()

In [ ]:
def get_probs(model, tokenizer, prompt, options, device):
    """Get softmax probabilities for each option."""
    texts = [str(prompt) + ' [SEP] ' + str(opt) for opt in options]
    enc = tokenizer(texts, padding=True, truncation=True, max_length=256, return_tensors='pt').to(device)
    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits.squeeze(), dim=-1)
    return probs.cpu().numpy()

## Q1: DeBERTa Single Model Inference (Row 25)

In [ ]:
row = test.iloc[25]
prompt = str(row['prompt'])
options = [str(row[l]) for l in LABELS]

deberta_probs = get_probs(deberta_model, deberta_tok, prompt, options, device)
best_idx = np.argmax(deberta_probs)
print(f'Highest: {idx2label[best_idx]}, {deberta_probs[best_idx]:.8f}')  # A, 0.29296875

## Q2-Q3: Simple & Weighted Ensembling

In [ ]:
roberta_probs = get_probs(roberta_model, roberta_tok, prompt, options, device)

# Q2: Simple average
avg_probs = (deberta_probs + roberta_probs) / 2
print('Simple ensemble top:', idx2label[np.argmax(avg_probs)])  # A

# Q3: Weighted (0.7 DeBERTa, 0.3 RoBERTa)
weighted = 0.7 * deberta_probs + 0.3 * roberta_probs
print('Weighted top:', idx2label[np.argmax(weighted)])  # A

## Q4-Q5: Top-3 Predictions & Full Submission

In [ ]:
# Q4: Top-3 for row 25
ranked = np.argsort(weighted)[::-1][:3]
pred_str = ' '.join([idx2label[i] for i in ranked])
print('Top-3:', pred_str)  # A D E

# Q5: Full submission
predictions = []
for i, row in test.iterrows():
    p = str(row['prompt'])
    opts = [str(row[l]) for l in LABELS]
    d_probs = get_probs(deberta_model, deberta_tok, p, opts, device)
    r_probs = get_probs(roberta_model, roberta_tok, p, opts, device)
    w = 0.7 * d_probs + 0.3 * r_probs
    top3 = np.argsort(w)[::-1][:3]
    predictions.append((row['id'], ' '.join([idx2label[j] for j in top3])))

sub = pd.DataFrame(predictions, columns=['id','prediction'])
sub.to_csv('submission.csv', index=False)
print('Prediction rows:', len(sub))  # 500

## Q6: Test-Time Augmentation

In [ ]:
tta_diffs = 0
for i in range(50):
    row = test.iloc[i]
    p = str(row['prompt'])
    opts = [str(row[l]) for l in LABELS]
    # Original
    orig_probs = get_probs(deberta_model, deberta_tok, p, opts, device)
    # Augmented
    aug_p = 'Answer the following multiple-choice question carefully: ' + p
    aug_probs = get_probs(deberta_model, deberta_tok, aug_p, opts, device)
    # Average
    tta_avg = (orig_probs + aug_probs) / 2
    if np.argmax(orig_probs) != np.argmax(tta_avg):
        tta_diffs += 1
print('TTA differences:', tta_diffs)  # 3

## Q7-Q9: DeBERTa vs Ensemble Comparison

In [ ]:
diff_top1 = 0
pos_gain = 0
rank_change = 0

for i in range(100):
    row = test.iloc[i]
    p, opts = str(row['prompt']), [str(row[l]) for l in LABELS]
    d = get_probs(deberta_model, deberta_tok, p, opts, device)
    r = get_probs(roberta_model, roberta_tok, p, opts, device)
    w = 0.7 * d + 0.3 * r
    
    # Q7: Different top-1
    if np.argmax(d) != np.argmax(w): diff_top1 += 1
    # Q8: Confidence gain
    if np.max(w) - np.max(d) > 0: pos_gain += 1
    # Q9: Top-3 ranking change
    d3 = ' '.join([idx2label[j] for j in np.argsort(d)[::-1][:3]])
    w3 = ' '.join([idx2label[j] for j in np.argsort(w)[::-1][:3]])
    if d3 != w3: rank_change += 1

print('Different top-1:', diff_top1)   # 5
print('Positive gain:', pos_gain)       # 0
print('Rank changes:', rank_change)     # 23